# Figure 2 (`make_figure_2.ipynb`)

**Figure 1** is omitted here by design.

**Figure 2** (below): one row of LS entropy-deficit curves \(\ln|D| - S\) vs \(\sigma_t^2\) (same style as `toy_gaussian_field_ls_els_bbels_entropy.ipynb` / 5×5 LS), for five datasets. Subpanels are labeled **(b)–(f)**. Outputs are written under `results/` as PNG and SVG.

Heavy computation is in its own cells so you can re-run plotting without recomputing.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import torch


def _project_root() -> Path:
    """Find repo root (contains scripts/sample_hierarchy_synthetic_images.py and src/)."""
    here = Path.cwd().resolve()
    seeds = [here]
    if here.name == "notebooks":
        seeds.append(here.parent)
    sub = here / "convolutional_diffusion"
    if sub.is_dir():
        seeds.append(sub.resolve())
    for start in seeds:
        for p in [start, *start.parents]:
            if (
                (p / "scripts" / "sample_hierarchy_synthetic_images.py").is_file()
                and (p / "src").is_dir()
            ):
                return p
    raise FileNotFoundError(
        "Could not find convolutional_diffusion repo root from cwd=%r. "
        "cd into the repo or notebooks/ under it, or set os.chdir('/path/to/convolutional_diffusion')."
        % (here,)
    )


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import importlib.util
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from src.utils.data import get_dataset
from src.utils.idealscore import LocalScoreModule
from src.utils.noise_schedules import cosine_noise_schedule

_hier_path = ROOT / "scripts" / "sample_hierarchy_synthetic_images.py"
_spec = importlib.util.spec_from_file_location("hier_synth", _hier_path)
_hier = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_hier)
sample_hierarchical_mixture_gaussian_images = _hier.sample_hierarchical_mixture_gaussian_images

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "device:", device)

ROOT: /home/users/hshunt/convolutional_diffusion device: cuda


## Figure 2 workflow

Each panel **(b)–(f)** now uses two cells:
1. **Data processing** (slow): compute entropy-deficit curves.
2. **Plotting** (fast): render that panel only.

A final cell combines all computed panel results into one multiplot figure and exports PNG/SVG.

In [3]:
# Shared settings
kernel_size_ls = 10
score_batch_size_ls = 64
t_grid_ls = np.linspace(0.02, 0.9, 10).tolist()
n_train_powers = [8, 10, 12, 14]
synthetic_train_size = 50_000
empty_cache_each_call = device.type == "cuda"

with torch.no_grad():
    t_tensor = torch.tensor(t_grid_ls, device=device, dtype=torch.float32)
    beta_t_grid = cosine_noise_schedule(t_tensor)
    snr_grid_ls = ((1.0 - beta_t_grid) / (beta_t_grid + 1e-12)).cpu().numpy()
    sigma_t2_grid_ls = 1.0 / (snr_grid_ls + 1e-12)

SIGMA_T2_MIN = 10 ** (-2.5)
SIGMA_T2_MAX = 10 ** 3

def make_dense_sigma_t2_grid(n_points=200):
    return np.logspace(np.log10(SIGMA_T2_MIN), np.log10(SIGMA_T2_MAX), int(n_points))

theory_sigma_t2_grid_ls = make_dense_sigma_t2_grid(n_points=200)


def posterior_entropy_nats_from_ls(ls_module, t, xt, dev):
    """Mean posterior entropy (nats) from LS; use module kernel size (do not hard-code k)."""
    out = ls_module.forward_with_posterior_stats(t, xt, device=dev)
    return float(out[2].mean().item())


def toeplitz_covariance_matrix(image_size=32, alpha=1.7, sigma_sq=1.0):
    """Construct Toeplitz-like covariance matrix Sigma for the synthetic field."""
    H = W = image_size
    Npix = H * W
    ys = torch.arange(H, dtype=torch.float64)
    xs = torch.arange(W, dtype=torch.float64)
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    coords = torch.stack([yy.reshape(-1), xx.reshape(-1)], dim=1)
    r = torch.cdist(coords, coords, p=2)
    r_safe = torch.clamp(r, min=1.0)
    if abs(alpha - 2.0) < 1e-6:
        C = sigma_sq * torch.log(r_safe)
    else:
        C = sigma_sq / (r_safe ** (2.0 - alpha))
    idx = torch.arange(Npix)
    C[idx, idx] = sigma_sq
    C = 0.5 * (C + C.T) + 1e-4 * torch.eye(Npix, dtype=C.dtype)
    return C


def build_toeplitz_like_dataset(n_samples, image_size=32, alpha=1.7, sigma_sq=1.0, seed=0):
    """Distance-kernel Gaussian field on a 2D grid (same construction as toy Toeplitz sanity cell)."""
    C = toeplitz_covariance_matrix(image_size=image_size, alpha=alpha, sigma_sq=sigma_sq)
    eigvals, eigvecs = torch.linalg.eigh(C)
    lam = torch.clamp(eigvals, min=1e-6)
    L = eigvecs @ torch.diag(torch.sqrt(lam))
    Npix = image_size * image_size
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_samples, Npix))
    L_np = L.detach().cpu().numpy()
    flat = z @ L_np.T
    # Single-channel (grayscale): matches scalar Toeplitz field / toy notebook; do not duplicate RGB.
    x = flat.reshape(n_samples, 1, image_size, image_size).astype(np.float32)
    t = torch.from_numpy(x).float()
    labels = torch.zeros(n_samples, dtype=torch.long)
    return TensorDataset(t, labels)


def build_hierarchical_mixture_dataset(
    n_samples,
    image_size=32,
    seed=1,
    num_channels=3,
    num_components=16,
    mu_0=1.0,
    sigma=0.15,
    return_meta=False,
):
    """Build hierarchical-mixture dataset and optionally return sampled means/params."""
    rng = np.random.default_rng(seed)
    means = rng.normal(
        loc=0.0,
        scale=mu_0,
        size=(num_components, num_channels, image_size, image_size),
    ).astype(np.float32)
    z = rng.integers(0, num_components, size=n_samples)
    eps = rng.standard_normal(
        size=(n_samples, num_channels, image_size, image_size),
        dtype=np.float32,
    )
    arr = means[z] + np.float32(sigma) * eps

    t = torch.from_numpy(arr).float()
    labels = torch.zeros(n_samples, dtype=torch.long)
    ds = TensorDataset(t, labels)

    if return_meta:
        return ds, {
            "means": torch.from_numpy(means).float(),
            "num_components": int(num_components),
            "sigma": float(sigma),
            "num_channels": int(num_channels),
            "image_size": int(image_size),
        }
    return ds


def compute_deficits_for_tensor_train(train_dataset, short_name, image_size, channels, device):
    max_available = len(train_dataset)
    n_train_list = sorted(
        {min(2**k, max_available) for k in n_train_powers}
    )
    eval_bs = min(64, max_available)
    idx = torch.randperm(max_available)[:eval_bs]
    xs = torch.stack([train_dataset[i][0] for i in idx]).to(device)

    deficit_by_n = {}
    for n_train in n_train_list:
        ls_n = (
            LocalScoreModule(
                train_dataset,
                kernel_size=kernel_size_ls,
                image_size=image_size,
                batch_size=score_batch_size_ls,
                schedule=cosine_noise_schedule,
                max_samples=n_train,
            )
            .to(device)
            .eval()
        )
        deficits = []
        for t_val in t_grid_ls:
            t = torch.tensor([t_val], device=device, dtype=torch.float32)
            beta_t = cosine_noise_schedule(t)
            at = torch.sqrt(torch.clamp(1.0 - beta_t, min=1e-12))
            bt = torch.sqrt(torch.clamp(beta_t, min=1e-12))
            eps = torch.randn_like(xs, device=device)
            xt = at * xs + bt * eps
            s_nats = posterior_entropy_nats_from_ls(ls_n, t, xt, device)
            deficits.append(float(np.log(n_train) - s_nats))
            if empty_cache_each_call:
                torch.cuda.empty_cache()
        deficit_by_n[n_train] = np.array(deficits)
        del ls_n
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return {
        "sigma_t2": np.array(sigma_t2_grid_ls),
        "deficit_by_n": {n: np.array(v) for n, v in deficit_by_n.items()},
        "n_train_list": list(n_train_list),
        "kernel_size": kernel_size_ls,
        "title": short_name,
        "dataset_label": short_name,
    }


def compute_deficits_real(ds_key, display_name, device):
    train_dataset, meta = get_dataset(ds_key, root=str(ROOT / "data"), train=True)
    eval_dataset, _ = get_dataset(ds_key, root=str(ROOT / "data"), train=False)
    image_size = int(meta["image_size"])
    channels = int(meta["num_channels"])
    max_available = len(train_dataset)
    n_train_list = sorted(
        {min(2**k, max_available) for k in n_train_powers}
    )
    eval_bs = min(64, len(eval_dataset))
    ev_loader = DataLoader(eval_dataset, batch_size=eval_bs, shuffle=False)
    eval_x0, _ = next(iter(ev_loader))
    eval_x0 = eval_x0.to(device)

    deficit_by_n = {}
    for n_train in n_train_list:
        ls_n = (
            LocalScoreModule(
                train_dataset,
                kernel_size=kernel_size_ls,
                image_size=image_size,
                batch_size=score_batch_size_ls,
                schedule=cosine_noise_schedule,
                max_samples=n_train,
            )
            .to(device)
            .eval()
        )
        deficits = []
        for t_val in t_grid_ls:
            t = torch.tensor([t_val], device=device, dtype=torch.float32)
            beta_t = cosine_noise_schedule(t)
            at = torch.sqrt(torch.clamp(1.0 - beta_t, min=1e-12))
            bt = torch.sqrt(torch.clamp(beta_t, min=1e-12))
            eps = torch.randn_like(eval_x0, device=device)
            xt = at * eval_x0 + bt * eps
            s_nats = posterior_entropy_nats_from_ls(ls_n, t, xt, device)
            deficits.append(float(np.log(n_train) - s_nats))
            if empty_cache_each_call:
                torch.cuda.empty_cache()
        deficit_by_n[n_train] = np.array(deficits)
        del ls_n
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # Real-data theory covariance uses all sliding patches from sampled images
    # (not a single fixed patch location such as top-left).
    Sigma_patch_real = patch_covariance_from_tensor_dataset(
        train_dataset,
        kernel_size=kernel_size_ls,
        max_images=512,
        max_patches=None,
        seed=0,
    )
    theory_by_n = gaussian_theory_deficits_from_cov(
        Sigma_patch_real,
        n_train_list,
        theory_sigma_t2_grid_ls,
    )

    return {
        "sigma_t2": np.array(sigma_t2_grid_ls),
        "theory_sigma_t2": np.array(theory_sigma_t2_grid_ls),
        "deficit_by_n": {n: np.array(v) for n, v in deficit_by_n.items()},
        "theory_by_n": {n: np.array(v) for n, v in theory_by_n.items()},
        "n_train_list": list(n_train_list),
        "kernel_size": kernel_size_ls,
        "title": display_name,
        "dataset_label": display_name,
    }


def patch_covariance_from_tensor_batch(
    images,
    kernel_size,
    max_images=512,
    max_patches=50000,
    seed=0,
):
    """Empirical patch covariance from tensor images (N, C, H, W).

    Uses all sliding patches (stride 1) from each sampled image, then optionally
    subsamples from that full patch pool for efficiency.
    """
    rng = np.random.default_rng(seed)
    n_total = int(images.shape[0])
    n_images = min(n_total, int(max_images))
    image_idx = rng.choice(n_total, size=n_images, replace=False)

    patch_rows = []
    for idx in image_idx:
        x = images[int(idx)]
        patches = x.unfold(1, kernel_size, 1).unfold(2, kernel_size, 1)
        patches = patches.contiguous().view(x.shape[0], -1, kernel_size, kernel_size)
        patches = patches.permute(1, 0, 2, 3).reshape(-1, x.shape[0] * kernel_size * kernel_size)
        patch_rows.append(patches)

    X = torch.cat(patch_rows, dim=0)
    if max_patches is not None and X.shape[0] > int(max_patches):
        keep = torch.from_numpy(rng.choice(X.shape[0], size=int(max_patches), replace=False))
        X = X[keep]

    X = X.to(torch.float64)
    X = X - X.mean(dim=0, keepdim=True)
    denom = max(int(X.shape[0]) - 1, 1)
    cov = (X.T @ X) / float(denom)
    cov = 0.5 * (cov + cov.T)
    return cov


def patch_covariance_from_tensor_dataset(
    dataset,
    kernel_size,
    max_images=512,
    max_patches=50000,
    seed=0,
):
    """Empirical patch covariance Sigma from a TensorDataset (channels x k x k patches)."""
    rng = np.random.default_rng(seed)
    n_images = min(len(dataset), int(max_images))
    image_idx = rng.choice(len(dataset), size=n_images, replace=False)
    imgs = torch.stack([dataset[int(i)][0] for i in image_idx], dim=0)
    return patch_covariance_from_tensor_batch(
        imgs,
        kernel_size=kernel_size,
        max_images=n_images,
        max_patches=max_patches,
        seed=seed,
    )


def gaussian_theory_deficits_from_cov(covariance, n_train_list, sigma_t2_grid):
    """Theory: min(log|D|, 0.5 * Tr[log(I + Sigma_patch / sigma_t^2)])."""
    eigvals = torch.linalg.eigvalsh(covariance).detach().cpu().numpy()
    eigvals = np.clip(eigvals, 0.0, None)

    theory_by_n = {}
    for n_train in n_train_list:
        cap = np.log(float(n_train))
        traces = []
        for sigma_t2 in sigma_t2_grid:
            val = 0.5 * np.log1p(eigvals / (float(sigma_t2) + 1e-12)).sum()
            traces.append(min(cap, float(val)))
        theory_by_n[n_train] = np.array(traces)
    return theory_by_n


def mixture_gaussian_theory_deficits_isotropic(
    sigma,
    mu_0,
    num_components,
    patch_dim,
    n_train_list,
    sigma_t2_grid,
):
    """Analytic HGM theory in patch space using dataset parameters (no empirical covariances).

    min(log|D|,
        log|M| + 0.5 Tr log(I + Sigma/sigma_t^2),
        0.5 Tr log(I + (Sigma + Sigma_M)/sigma_t^2))
    with Sigma = sigma^2 I and Sigma_M = mu_0^2 I.
    """
    sigma2 = float(sigma) ** 2
    mu02 = float(mu_0) ** 2
    d = float(patch_dim)
    log_m = np.log(float(num_components))

    theory_by_n = {}
    for n_train in n_train_list:
        cap = np.log(float(n_train))
        vals = []
        for sigma_t2 in sigma_t2_grid:
            s2 = float(sigma_t2) + 1e-12
            term2 = log_m + 0.5 * d * np.log1p(sigma2 / s2)
            term3 = 0.5 * d * np.log1p((sigma2 + mu02) / s2)
            vals.append(min(cap, float(term2), float(term3)))
        theory_by_n[n_train] = np.array(vals)
    return theory_by_n


def plot_entropy_deficit_panel(ax, letter, info, ds_name):
    ax.text(
        -0.12,
        1.05,
        f"({letter})",
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="bottom",
        ha="left",
    )
    if info is None:
        ax.set_axis_off()
        ax.text(
            0.5,
            0.5,
            "Random hierarchy\n(not implemented)",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=10,
        )
        ax.set_title(f"{ds_name}", fontsize=10)
        return

    sigma_t2_exp = info["sigma_t2"]
    sigma_t2_theory = info.get("theory_sigma_t2", sigma_t2_exp)
    for n_train in info["n_train_list"]:
        line = ax.plot(
            sigma_t2_exp,
            info["deficit_by_n"][n_train],
            linestyle="--",
            linewidth=0.5,
            marker="o",
            label=f"|D|={n_train}",
            markersize=4,
        )[0]
        if "theory_by_n" in info and n_train in info["theory_by_n"]:
            ax.plot(
                sigma_t2_theory,
                info["theory_by_n"][n_train],
                linestyle="-",
                linewidth=2.0,
                alpha=0.95,
                color=line.get_color(),
                zorder=line.get_zorder() + 1,
            )
    ax.set_xscale("log")
    ax.set_xlim(SIGMA_T2_MIN, SIGMA_T2_MAX)
    ax.set_xlabel(r"$\sigma_t^2$")
    ax.set_ylabel(r"$\ln|D|-S$ (nats)")
    ax.set_title(f"{ds_name}", fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=7)


def preview_single_panel(letter, info, ds_name, panel_size=3.3):
    fig, ax = plt.subplots(1, 1, figsize=(panel_size, panel_size))
    plot_entropy_deficit_panel(ax, letter, info, ds_name)
    plt.tight_layout()
    plt.show()

In [4]:
# (b) Toeplitz field — data processing (slow)
print("Toeplitz-like field…")
ds_t = build_toeplitz_like_dataset(synthetic_train_size, image_size=32, seed=0)
panel_b_info = compute_deficits_for_tensor_train(ds_t, "Toeplitz field", 32, 1, device)

# Add dotted theory curves using PATCH covariance Sigma_patch.
Sigma_patch_b = patch_covariance_from_tensor_dataset(
    ds_t,
    kernel_size=kernel_size_ls,
    max_images=512,
    max_patches=50000,
    seed=0,
)
panel_b_info["theory_sigma_t2"] = np.array(theory_sigma_t2_grid_ls)
panel_b_info["theory_by_n"] = gaussian_theory_deficits_from_cov(
    Sigma_patch_b,
    panel_b_info["n_train_list"],
    panel_b_info["theory_sigma_t2"],
)

Toeplitz-like field…


RuntimeError: shape '[64, 64, 100, 32, 32]' is invalid for input of size 446054400

# (b) Toeplitz field — plotting (fast)

In [ ]:
# Ensure theory curves exist even if previous cells were rerun out of order.
if "theory_by_n" not in panel_b_info:
    Sigma_patch_b = patch_covariance_from_tensor_dataset(
        ds_t,
        kernel_size=kernel_size_ls,
        max_images=512,
        max_patches=50000,
        seed=0,
    )
    panel_b_info["theory_sigma_t2"] = np.array(theory_sigma_t2_grid_ls)
    panel_b_info["theory_by_n"] = gaussian_theory_deficits_from_cov(
        Sigma_patch_b,
        panel_b_info["n_train_list"],
        panel_b_info["theory_sigma_t2"],
    )

print("(b) keys:", sorted(panel_b_info.keys()))
preview_single_panel("b", panel_b_info, "Toeplitz field")

In [ ]:
# (c) Hierarchical mixture of Gaussians — data processing (slow)
print("Hierarchical mixture of Gaussians…")
hm_params = {
    "num_channels": 3,
    "num_components": 16,
    "mu_0": 1.0,
    "sigma": 0.15,
    "image_size": 32,
    "seed": 1,
}
ds_h = build_hierarchical_mixture_dataset(
    synthetic_train_size,
    image_size=hm_params["image_size"],
    seed=hm_params["seed"],
    num_channels=hm_params["num_channels"],
    num_components=hm_params["num_components"],
    mu_0=hm_params["mu_0"],
    sigma=hm_params["sigma"],
)
panel_c_info = compute_deficits_for_tensor_train(
    ds_h, "Hierarchical mixture of Gaussians", 32, 3, device
)

# Add analytic mixture-of-Gaussians theory from dataset parameters.
patch_dim = hm_params["num_channels"] * kernel_size_ls * kernel_size_ls
panel_c_info["theory_sigma_t2"] = np.array(theory_sigma_t2_grid_ls)
panel_c_info["theory_by_n"] = mixture_gaussian_theory_deficits_isotropic(
    sigma=hm_params["sigma"],
    mu_0=hm_params["mu_0"],
    num_components=hm_params["num_components"],
    patch_dim=patch_dim,
    n_train_list=panel_c_info["n_train_list"],
    sigma_t2_grid=panel_c_info["theory_sigma_t2"],
)

In [ ]:
# (c) Hierarchical mixture of Gaussians — plotting (fast)
if "theory_by_n" not in panel_c_info:
    print("Adding analytic hierarchical-mixture theory curves...")
    hm_params = {
        "num_channels": 3,
        "num_components": 16,
        "mu_0": 1.0,
        "sigma": 0.15,
    }
    patch_dim = hm_params["num_channels"] * kernel_size_ls * kernel_size_ls
    panel_c_info["theory_sigma_t2"] = np.array(theory_sigma_t2_grid_ls)
    panel_c_info["theory_by_n"] = mixture_gaussian_theory_deficits_isotropic(
        sigma=hm_params["sigma"],
        mu_0=hm_params["mu_0"],
        num_components=hm_params["num_components"],
        patch_dim=patch_dim,
        n_train_list=panel_c_info["n_train_list"],
        sigma_t2_grid=panel_c_info["theory_sigma_t2"],
    )

print("(c) keys:", sorted(panel_c_info.keys()))
preview_single_panel("c", panel_c_info, "Hierarchical Mixture of Gaussians")

In [ ]:
# (d) Random hierarchy model — data processing (slow)
print("Random hierarchy model…")
try:
    _rh = _hier.sample_random_hierarchy_images(
        synthetic_train_size,
        image_size=32,
        num_channels=3,
        seed=2,
    )
    ds_r = TensorDataset(
        torch.from_numpy(_rh).float(),
        torch.zeros(synthetic_train_size, dtype=torch.long),
    )
    panel_d_info = compute_deficits_for_tensor_train(
        ds_r, "Random hierarchy model", 32, 3, device
    )
except NotImplementedError as e:
    print("  (skipped)", e)
    panel_d_info = None

In [ ]:
# (d) Random hierarchy model — plotting (fast)
preview_single_panel("d", panel_d_info, "Random hierarchy model")

In [ ]:
# (e) CIFAR-10 — data processing (slow)
print("CIFAR-10…")
panel_e_info = compute_deficits_real("cifar10", "CIFAR-10", device)

In [ ]:
# (e) CIFAR-10 — plotting (fast)
if "theory_by_n" not in panel_e_info:
    print("Recomputing CIFAR-10 panel to include theory curves...")
    panel_e_info = compute_deficits_real("cifar10", "CIFAR-10", device)

print("(e) keys:", sorted(panel_e_info.keys()))
preview_single_panel("e", panel_e_info, "CIFAR-10")

In [ ]:
# (f) CelebA — data processing (slow)
print("CelebA…")
panel_f_info = compute_deficits_real("celeba", "CelebA", device)

In [ ]:
# (f) CelebA — plotting (fast)
if "theory_by_n" not in panel_f_info:
    print("Recomputing CelebA panel to include theory curves...")
    panel_f_info = compute_deficits_real("celeba", "CelebA", device)

print("(f) keys:", sorted(panel_f_info.keys()))
preview_single_panel("f", panel_f_info, "CelebA")

In [ ]:
# Final: combine selected panels into one 4-plot figure and export
from matplotlib.lines import Line2D

save_dir = ROOT / "results"
save_dir.mkdir(parents=True, exist_ok=True)

fig2_row = [
    ("a", panel_b_info, "Power Law Gaussian Data"),
    ("b", panel_c_info, "Hierarchical Mixture of Gaussians"),
    ("c", panel_e_info, "CIFAR-10"),
    ("d", panel_f_info, "CelebA"),
]

sq = 3.3
fig, axes = plt.subplots(1, 4, figsize=(4 * sq, sq), sharex=True, sharey=True)
fig.suptitle(
    "LS average entropy deficit vs $\\sigma_t^2$ (10×10 kernel)",
    fontsize=11,
    y=1.04,
)

for ax, (letter, info, ds_name) in zip(axes, fig2_row):
    plot_entropy_deficit_panel(ax, letter, info, ds_name)
    ax.set_xlabel("")
    ax.set_ylabel("")
    lg = ax.get_legend()
    if lg is not None:
        lg.remove()

# Build one shared legend: theory/experiment style + dataset sizes (powers of 2).
exp_handles, exp_labels = axes[0].get_legend_handles_labels()
size_handles = []
for h, label in zip(exp_handles, exp_labels):
    if label.startswith("|D|="):
        n = int(label.split("=")[1])
        k = int(round(np.log2(n)))
        pretty = rf"$|D|=2^{{{k}}}$"
        size_handles.append(Line2D([0], [0], color=h.get_color(), linestyle="-", linewidth=2, label=pretty))

shared_handles = [
    Line2D([0], [0], color="black", linestyle="-", linewidth=2.0, label="Theory"),
    Line2D([0], [0], color="black", linestyle="--", linewidth=0.5, marker="o", markersize=5, label="Experiment"),
] + size_handles

fig.legend(
    handles=shared_handles,
    loc="upper center",
    ncol=min(len(shared_handles), 6),
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
)

fig.supxlabel(r"$\sigma_t^2$")
fig.supylabel(r"$\ln|D|-S$ (nats)")

plt.tight_layout(rect=(0, 0, 1, 0.90))
for ext in ("png", "svg"):
    out = save_dir / f"figure2_entropy_deficit_row_ad.{ext}"
    fig.savefig(out, format=ext, bbox_inches="tight", dpi=200)
    print("Saved", out)
plt.show()

In [ ]:
# (optional) end-of-notebook marker
print("Figure 2 notebook restructured into panel process/plot pairs.")